# 🐝 Lab 2.2 — Optimizer Arena: SGD → AdamW → Muon

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varoonsahgal/advancedml/blob/main/notebooks/lab05_optimizer_arena.ipynb)

**⏱ Time:** ~2 hours  **🐝 Referee:** Buzz the Backprop Bee

### 🎯 Learning objectives
- Implement **SGD, Momentum, Nesterov, Adam, AdamW, Lion and Muon** from scratch as real `torch.optim.Optimizer` subclasses — and prove your AdamW matches `torch.optim.AdamW`.
- Explain *why* **decoupled weight decay** (AdamW) differs from L2 regularization inside Adam.
- Understand **Muon**: momentum + **Newton–Schulz orthogonalization** of matrix-shaped updates, i.e. steepest descent under the spectral norm.
- Race optimizers on **Rosenbrock / Beale** and on **FashionMNIST**, with fair per-optimizer learning-rate tuning.
- Build **warmup+cosine** and **WSD (warmup-stable-decay)** schedules and run an **LR range test**.
- Explain why **residual connections + normalization (LayerNorm/RMSNorm) + sane init** make deep nets trainable.

### 🧰 Prerequisites
Gradients & backprop (Lab 2.1), a PyTorch training loop, basic linear algebra (SVD).

### 🌍 Why this matters in 2026
The optimizer is the engine of every training run, and it is *still* an active frontier. AdamW has been the default for LLMs since ~2019, but in late 2024 **Muon** set speed records on NanoGPT and CIFAR-10 speedruns, and in 2025 it was shown to scale to multi-billion-parameter LLMs (Moonshot AI's Moonlight, then Kimi K2 with "MuonClip"). Meanwhile **WSD schedules** became popular because they let you stop (or continue) training at any point without restarting a cosine schedule. Knowing how these work at the level of individual tensor ops lets you read new optimizer papers in an afternoon — and spot when a "new" method is an old one in a new norm. 🍯

## 🗺️ Roadmap

| Part | What you'll do | ⏱ min |
|---|---|---|
| 0 | Setup | 5 |
| 1 | Meet the arena: Rosenbrock and Beale | 5 |
| 2 | Build SGD/Momentum/Nesterov, Adam/AdamW (🧪 #1), Lion (🧪 #2), Newton–Schulz (🧪 #3) and Muon | 40 |
| 3 | Race #1 on 2-D landscapes with per-optimizer LR tuning — plus an animated race 🎬 | 15 |
| 4 | Race #2: an MLP on FashionMNIST | 15 |
| 5 | LR schedules: warmup+cosine vs WSD (🧪 #4) | 15 |
| 6 | The LR range test | 10 |
| 7 | Norms, residuals and init: RMSNorm (🧪 #5) and a 48-block depth experiment | 15 |

## 0. Setup
FashionMNIST (~30 MB) downloads via `torchvision` (preinstalled on Colab). Everything runs on CPU; a GPU just makes Part 4 faster.

In [ ]:
import os, math, random, time
import numpy as np
import torch
import matplotlib.pyplot as plt

FAST = os.environ.get("LAB_FAST", "0") == "1"   # smoke-test mode (normal users: leave unset)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(0)

HIVE = ["#F5B700", "#1F1B16", "#FF6B5B", "#1FA39A", "#3D4DB7", "#9B5DE5"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=HIVE), "figure.facecolor": "#FFFBEA",
                     "axes.facecolor": "#FFFDF5", "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 11})
print(f"torch {torch.__version__} | device={DEVICE} | FAST={FAST}")

import copy
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Optimizer

DATA_DIR = os.path.join(os.path.expanduser("~"), ".cache", "hive_data")   # outside the repo
os.makedirs(DATA_DIR, exist_ok=True)

def ema(xs, beta=0.95):
    """Exponential moving average for readable loss curves."""
    out, m = [], None
    for x in xs:
        m = x if m is None else beta * m + (1 - beta) * x
        out.append(m)
    return np.array(out)

## Part 1 — The arena: two famously nasty 2-D landscapes 🏟️

Before touching neural nets, let's watch optimizers on functions we can *see*.

**Rosenbrock** ("the banana"): $f(x,y) = (1-x)^2 + 100\,(y - x^2)^2$, minimum $f(1,1)=0$. The valley floor is easy to find but curved and nearly flat along its length, while the walls are ~100× steeper — a perfect cartoon of the **ill-conditioning** in real loss surfaces.

**Beale**: $f(x,y) = (1.5 - x + xy)^2 + (2.25 - x + xy^2)^2 + (2.625 - x + xy^3)^2$, minimum $f(3, 0.5) = 0$. Flat plateaus and sharp ridges — gradient *magnitudes* vary wildly, which is exactly where adaptive and sign-based methods shine.

In [ ]:
def rosenbrock(x, y): return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2
def beale(x, y):      return (1.5 - x + x * y) ** 2 + (2.25 - x + x * y ** 2) ** 2 + (2.625 - x + x * y ** 3) ** 2

ARENAS = {
    "Rosenbrock": dict(f=rosenbrock, start=(-1.5, 2.0), opt=(1.0, 1.0), xlim=(-2, 2), ylim=(-1, 3)),
    "Beale":      dict(f=beale,      start=(1.0, 1.5),  opt=(3.0, 0.5), xlim=(-0.5, 4), ylim=(-1.5, 2)),
}

def plot_landscape(ax, name):
    A = ARENAS[name]
    xs, ys = np.linspace(*A["xlim"], 300), np.linspace(*A["ylim"], 300)
    X, Y = np.meshgrid(xs, ys)
    Z = A["f"](X, Y)
    ax.contourf(X, Y, np.log10(Z + 1e-3), levels=30, cmap="YlOrBr_r", alpha=0.85)
    ax.contour(X, Y, np.log10(Z + 1e-3), levels=15, colors="#1F1B16", linewidths=0.3, alpha=0.5)
    ax.plot(*A["opt"], marker="*", ms=16, color=HIVE[2], mec="k", zorder=5)
    ax.plot(*A["start"], marker="o", ms=8, color="white", mec="k", zorder=5)
    ax.set_title(f"{name} (log10 f)"); ax.set_xlim(A["xlim"]); ax.set_ylim(A["ylim"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, name in zip(axes, ARENAS): plot_landscape(ax, name)
plt.show()

> **🍯 Key takeaway:** Real loss surfaces are ill-conditioned: steep in some directions, nearly flat in others. Optimizers differ mainly in how they cope with that mismatch.

## Part 2 — Building the optimizers 🔧

A `torch.optim.Optimizer` holds `param_groups` (lists of tensors + hyperparameters) and a per-parameter `state` dict. All we write is `step()`. Notation: $g_t$ = gradient, $\eta$ = learning rate, $\theta$ = parameters.

### 2a. SGD, Momentum, Nesterov
$$\textbf{SGD:}\quad \theta \leftarrow \theta - \eta g_t$$
$$\textbf{Momentum (heavy ball):}\quad b_t = \mu b_{t-1} + g_t,\qquad \theta \leftarrow \theta - \eta\, b_t$$
$$\textbf{Nesterov:}\quad b_t = \mu b_{t-1} + g_t,\qquad \theta \leftarrow \theta - \eta\,(g_t + \mu b_t)$$

*Intuition:* momentum is a low-pass filter on gradients — directions that agree step-to-step (the valley floor) accumulate speed up to $1/(1-\mu)\times$; directions that flip sign (across the valley walls) cancel. Nesterov "looks ahead": it uses the momentum it is *about* to have, which damps overshoot. (The form above is the one PyTorch uses; it's algebraically equivalent to the classic "gradient at the look-ahead point" form after a change of variables.)

In [ ]:
class MySGD(Optimizer):
    def __init__(self, params, lr=1e-2, momentum=0.0, nesterov=False, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, mu, wd = group["lr"], group["momentum"], group["weight_decay"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                if wd:
                    g = g.add(p, alpha=wd)            # classic (coupled) L2: add wd*theta to the gradient
                if mu:
                    state = self.state[p]
                    if "buf" not in state:
                        state["buf"] = g.clone()      # PyTorch convention: first buffer = first gradient
                    else:
                        state["buf"].mul_(mu).add_(g)
                    g = g.add(state["buf"], alpha=mu) if group["nesterov"] else state["buf"]
                p.add_(g, alpha=-lr)
        return loss

### ✔️ Checkpoint — `MySGD` vs `torch.optim.SGD`
We run both on identical copies of a small network for 50 steps; parameters must agree. We use **float64** for these comparisons: in float32, different-but-equivalent operation orders give ~1e-7 rounding differences that training dynamics can amplify over hundreds of steps, which would hide whether the *algorithm* is right.

In [ ]:
def tiny_net(seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(10, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 3)).double()

def compare_optimizers(make_ours, make_ref, steps=50, seed=0):
    torch.manual_seed(123)
    X, y = torch.randn(64, 10, dtype=torch.float64), torch.randint(0, 3, (64,))
    m1, m2 = tiny_net(seed), tiny_net(seed)
    o1, o2 = make_ours(m1.parameters()), make_ref(m2.parameters())
    for _ in range(steps):
        for m, o in ((m1, o1), (m2, o2)):
            o.zero_grad(); F.cross_entropy(m(X), y).backward(); o.step()
    return max((p1 - p2).abs().max().item() for p1, p2 in zip(m1.parameters(), m2.parameters()))

for kw in [dict(), dict(momentum=0.9), dict(momentum=0.9, nesterov=True), dict(momentum=0.9, weight_decay=1e-2)]:
    d = compare_optimizers(lambda p: MySGD(p, lr=0.05, **kw), lambda p: torch.optim.SGD(p, lr=0.05, **kw))
    print(f"SGD {kw}: max |Δθ| = {d:.2e}")
    assert d < 1e-10
print("✔️ MySGD matches torch.optim.SGD")

### 2b. Adam and AdamW — and why "W" matters

**Adam** keeps EMAs of the gradient and of its square, with bias correction (the EMAs start at 0):
$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t,\qquad v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$
$$\hat m_t = \frac{m_t}{1-\beta_1^t},\quad \hat v_t = \frac{v_t}{1-\beta_2^t},\qquad \theta \leftarrow \theta - \eta\,\frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon}$$

Each coordinate gets its own effective step size $\eta/\sqrt{\hat v}$: a cheap **diagonal preconditioner**. Roughly, the update size per coordinate is ~$\eta$ regardless of the gradient's scale.

**The weight-decay subtlety (Loshchilov & Hutter).** With "L2 in Adam" you add $\lambda\theta$ to $g_t$ — but then the decay term *also gets divided by* $\sqrt{\hat v}$, so weights with large historical gradients are decayed *less*. That's not what we want from regularization. **AdamW** *decouples* it: shrink the weights directly, outside the adaptive machinery:
$$\theta \leftarrow \theta - \eta\lambda\,\theta \;-\; \eta\,\frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon}.$$
For plain SGD the two are identical (up to reparametrizing $\lambda$); for adaptive methods they are genuinely different algorithms.

### 🧪 Your Turn #1 — implement Adam / AdamW
Fill in the TODOs in `step()`. One class handles both: `decoupled=False` → L2 added to the gradient (like `torch.optim.Adam(weight_decay=...)`), `decoupled=True` → AdamW.

**Hints**
- State per parameter: `step` (int), `m` and `v` (zeros like `p`).
- In-place ops keep it fast: `m.mul_(b1).add_(g, alpha=1-b1)`, `v.mul_(b2).addcmul_(g, g, value=1-b2)`.
- Decoupled decay happens **before** the Adam update: `p.mul_(1 - lr*wd)`.
- Final update: `p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)`.

In [ ]:
# 🧪 YOUR TURN
class MyAdam(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0, decoupled=False):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay, decoupled=decoupled))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            lr, (b1, b2), eps, wd = group["lr"], group["betas"], group["eps"], group["weight_decay"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if not state:
                    state["step"], state["m"], state["v"] = 0, torch.zeros_like(p), torch.zeros_like(p)
                state["step"] += 1
                # TODO 1: weight decay — coupled (add wd*p to g) or decoupled (shrink p in place)
                # TODO 2: update the first and second moment EMAs
                # TODO 3: bias-correct and apply the update
                pass

def MyAdamW(params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-2):   # (given) AdamW = decoupled MyAdam
    return MyAdam(params, lr=lr, betas=betas, eps=eps, weight_decay=weight_decay, decoupled=True)

In [ ]:
# ✅ Solution (peek only after trying!)
class MyAdam(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0, decoupled=False):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay, decoupled=decoupled))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, (b1, b2), eps, wd = group["lr"], group["betas"], group["eps"], group["weight_decay"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if not state:
                    state["step"], state["m"], state["v"] = 0, torch.zeros_like(p), torch.zeros_like(p)
                state["step"] += 1
                t, m, v = state["step"], state["m"], state["v"]
                if wd:
                    if group["decoupled"]:
                        p.mul_(1 - lr * wd)                 # AdamW: decay the weights directly
                    else:
                        g = g.add(p, alpha=wd)              # Adam + L2: decay goes through the preconditioner
                m.mul_(b1).add_(g, alpha=1 - b1)
                v.mul_(b2).addcmul_(g, g, value=1 - b2)
                m_hat = m / (1 - b1 ** t)
                v_hat = v / (1 - b2 ** t)
                p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)
        return loss

def MyAdamW(params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-2):
    return MyAdam(params, lr=lr, betas=betas, eps=eps, weight_decay=weight_decay, decoupled=True)

### ✔️ Checkpoint — your AdamW vs `torch.optim.AdamW`
This is the headline checkpoint of the lab. 200 steps, strong weight decay; in float64 both must agree to ~1e-10.

In [ ]:
d_adam = compare_optimizers(lambda p: MyAdam(p, lr=3e-3, weight_decay=0.1),
                            lambda p: torch.optim.Adam(p, lr=3e-3, weight_decay=0.1), steps=200)
d_adamw = compare_optimizers(lambda p: MyAdamW(p, lr=3e-3, weight_decay=0.1, betas=(0.9, 0.95)),
                             lambda p: torch.optim.AdamW(p, lr=3e-3, weight_decay=0.1, betas=(0.9, 0.95)), steps=200)
d_cross = compare_optimizers(lambda p: MyAdamW(p, lr=3e-3, weight_decay=0.1),
                             lambda p: torch.optim.Adam(p, lr=3e-3, weight_decay=0.1), steps=200)
print(f"MyAdam  vs torch Adam : max |Δθ| = {d_adam:.2e}")
print(f"MyAdamW vs torch AdamW: max |Δθ| = {d_adamw:.2e}")
print(f"(sanity) AdamW vs Adam+L2 — genuinely different algorithms: max |Δθ| = {d_cross:.2e}")
assert d_adam < 1e-10 and d_adamw < 1e-10
assert d_cross > 1e-3
print("✔️ Your AdamW matches torch.optim.AdamW. 🐝")

### 2c. Lion — "EvoLved Sign Momentum"
Discovered by a program search over update rules (Chen et al., 2023). It keeps only **one** momentum buffer and outputs the **sign** of an interpolation:
$$c_t = \beta_1 m_{t-1} + (1-\beta_1) g_t,\qquad \theta \leftarrow \theta - \eta\,\big(\mathrm{sign}(c_t) + \lambda\theta\big),\qquad m_t = \beta_2 m_{t-1} + (1-\beta_2) g_t$$
with $\beta_1 = 0.9,\ \beta_2 = 0.99$ by default. Every coordinate moves by exactly $\pm\eta$, so the update has a larger norm than Adam's — use a **3–10× smaller learning rate** (and a correspondingly larger $\lambda$). Half the optimizer memory of Adam.

### 🧪 Your Turn #2 — implement Lion
**Hints:** `c = m.mul(b1).add(g, alpha=1-b1)` (not in place — you still need the old `m`), then `p.mul_(1-lr*wd)`, `p.add_(c.sign(), alpha=-lr)`, then update `m` in place with `b2`.

In [ ]:
# 🧪 YOUR TURN
class Lion(Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        super().__init__(params, dict(lr=lr, betas=betas, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                # TODO: Lion update (see hints)
                pass

In [ ]:
# ✅ Solution (peek only after trying!)
class Lion(Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        super().__init__(params, dict(lr=lr, betas=betas, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, (b1, b2), wd = group["lr"], group["betas"], group["weight_decay"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if "m" not in state:
                    state["m"] = torch.zeros_like(p)
                m = state["m"]
                c = m.mul(b1).add(g, alpha=1 - b1)     # interpolation used for the update
                p.mul_(1 - lr * wd)
                p.add_(c.sign(), alpha=-lr)
                m.mul_(b2).add_(g, alpha=1 - b2)       # momentum tracks with the slower beta2
        return loss

In [ ]:
# ✔️ Checkpoint — after one step from m=0, Lion moves every coordinate by exactly lr (when wd=0)
torch.manual_seed(0)
w = nn.Parameter(torch.randn(5, 5)); w0 = w.detach().clone()
opt = Lion([w], lr=0.01); (w ** 3).sum().backward(); opt.step()
assert torch.allclose((w - w0).abs(), torch.full_like(w, 0.01), atol=1e-6)
print("✔️ Lion takes ±lr steps — a sign-based optimizer!")

### 2d. Muon — orthogonalized momentum for matrices 🧲

**The idea (Jordan et al., 2024).** For a weight *matrix* $W\in\mathbb{R}^{m\times n}$, SGD-momentum updates tend to be **very low-rank**: a few singular directions dominate, so most directions barely learn. Muon replaces the momentum matrix $B = U\Sigma V^\top$ by its **orthogonal polar factor**
$$\mathrm{Orth}(B) = U V^\top,$$
i.e. it sets *all singular values to 1*, amplifying the rare-but-useful directions. Theoretically this is **steepest descent under the spectral norm** (Bernstein & Newhouse, 2024): the update $\Delta W$ that maximizes $\langle G, \Delta W\rangle$ subject to $\|\Delta W\|_{2} \le \eta$ is exactly $\eta\, UV^\top$. (Sign descent — Lion — is the analogous answer for the $\ell_\infty$ norm.)

**Computing $UV^\top$ fast — Newton–Schulz.** An SVD per step is slow on GPUs. Instead, normalize $X_0 = B/\|B\|_F$ (so all singular values are ≤ 1) and iterate an odd matrix polynomial that pushes every singular value toward 1:
$$X_{k+1} = a X_k + b\,(X_k X_k^\top) X_k + c\,(X_k X_k^\top)^2 X_k,\qquad (a,b,c) = (3.4445,\,-4.7750,\,2.0315).$$
Because the polynomial is odd, it acts on singular values only: $\sigma \mapsto a\sigma + b\sigma^3 + c\sigma^5$, and $U, V$ are untouched. The coefficients are tuned for a *steep slope at 0* (tiny singular values grow fast) rather than exact convergence, so after 5 steps singular values land roughly in $[0.7, 1.2]$ — good enough, and only matmuls, so it runs happily in bfloat16.

**Practical recipe.**
- Use Muon only for **2-D hidden weight matrices**. Embeddings, the output head, biases and norm gains go to AdamW.
- Shape-dependent scaling of the update: `max(1, rows/cols)**0.5` (Keller Jordan's version).
- **Scaling to LLMs (Liu et al., 2025, "Muon is Scalable for LLM Training"):** add decoupled weight decay, and rescale updates to $0.2\sqrt{\max(m,n)}\cdot UV^\top$ so their RMS matches AdamW's typical update RMS — then you can reuse AdamW's learning rate and weight decay. With this they reported ~2× compute efficiency vs AdamW and trained the Moonlight MoE model; Kimi K2 (2025) later used a Muon variant ("MuonClip") at the trillion-parameter scale.

### 🧪 Your Turn #3 — Newton–Schulz orthogonalization
Implement `newton_schulz5(G, steps=5)`.

**Hints:** (1) if `rows > cols`, work on `G.T` (so $XX^\top$ is the *smaller* Gram matrix) and transpose back at the end; (2) divide by the Frobenius norm (+1e-7); (3) loop: `A = X @ X.T; B = b*A + c*A@A; X = a*X + B@X`.

In [ ]:
# 🧪 YOUR TURN
def newton_schulz5(G, steps=5, eps=1e-7):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float()
    # TODO: transpose if tall, normalize, iterate, transpose back
    return X

In [ ]:
# ✅ Solution (peek only after trying!)
def newton_schulz5(G, steps=5, eps=1e-7):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float()
    transposed = X.size(0) > X.size(1)
    if transposed:
        X = X.T
    X = X / (X.norm() + eps)                     # all singular values now <= 1
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A
        X = a * X + B @ X                        # odd polynomial in X: touches only singular values
    return X.T if transposed else X

In [ ]:
# ✔️ Checkpoint + visualization: what does Newton–Schulz do to singular values?
torch.manual_seed(0)
# a realistic "gradient-like" matrix: low-rank-ish with a decaying spectrum
U_, _ = torch.linalg.qr(torch.randn(128, 128)); V_, _ = torch.linalg.qr(torch.randn(256, 128))
G = U_ @ torch.diag(torch.logspace(0, -3, 128)) @ V_.T

O = newton_schulz5(G)
s_before = torch.linalg.svdvals(G); s_after = torch.linalg.svdvals(O)
U, S, Vh = torch.linalg.svd(G, full_matrices=False)
cos_to_polar = (O * (U @ Vh)).sum() / (O.norm() * (U @ Vh).norm())
in_band = ((s_after > 0.5) & (s_after < 1.3)).float().mean().item()
print(f"singular values after NS: min {s_after.min():.3f}, max {s_after.max():.3f}; "
      f"{in_band:.0%} in [0.5, 1.3]; cosine to exact UV^T = {cos_to_polar:.4f}")
# Singular values spanning 3 orders of magnitude: the largest ~80% get pushed into the band;
# the very tiniest (<~1e-3 of the top after normalization) need more than 5 steps — an accepted trade-off.
assert s_after.max() < 1.3 and in_band > 0.75 and cos_to_polar > 0.9
print("✔️ Newton–Schulz orthogonalizes (approximately) — and it's only matmuls!")

sig = np.linspace(0, 1.2, 400); y_ = sig.copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for k in range(1, 6):
    y_ = 3.4445 * y_ - 4.7750 * y_ ** 3 + 2.0315 * y_ ** 5
    axes[0].plot(sig, y_, label=f"after {k} step{'s' * (k > 1)}", color=plt.cm.YlOrBr(0.3 + 0.14 * k))
axes[0].axhline(1, color="k", ls="--", lw=0.8); axes[0].set_xlabel("input singular value σ (after normalization)")
axes[0].set_ylabel("output"); axes[0].set_title("NS quintic iteration on a singular value"); axes[0].legend(fontsize=8)
axes[1].semilogy(s_before / s_before.max(), label="momentum matrix (normalized)", color=HIVE[1])
axes[1].semilogy(s_after, label="after Newton–Schulz", color=HIVE[0], lw=2.5)
axes[1].set_xlabel("singular value index"); axes[1].set_title("Spectrum flattening"); axes[1].legend()
plt.tight_layout(); plt.show()

Now the full optimizer. Like the reference implementation, one `Muon` object handles two kinds of parameter groups: `use_muon=True` groups get orthogonalized momentum; the rest get AdamW. We support both update scalings (`"keller"` and the 2025 `"moonlight"` RMS-matching rule).

In [ ]:
def adamw_update_(p, state, lr, betas, eps, wd):
    """Functional AdamW step used for Muon's non-matrix parameters."""
    b1, b2 = betas
    if not state:
        state["step"], state["m"], state["v"] = 0, torch.zeros_like(p), torch.zeros_like(p)
    state["step"] += 1
    t, m, v = state["step"], state["m"], state["v"]
    p.mul_(1 - lr * wd)
    m.mul_(b1).add_(p.grad, alpha=1 - b1)
    v.mul_(b2).addcmul_(p.grad, p.grad, value=1 - b2)
    p.addcdiv_(m / (1 - b1 ** t), (v / (1 - b2 ** t)).sqrt().add_(eps), value=-lr)

class Muon(Optimizer):
    """Muon for 2-D weights (groups with use_muon=True) + AdamW for everything else."""
    def __init__(self, param_groups):
        for g in param_groups:
            if g["use_muon"]:
                g.setdefault("lr", 0.02); g.setdefault("momentum", 0.95); g.setdefault("nesterov", True)
                g.setdefault("weight_decay", 0.0); g.setdefault("ns_steps", 5); g.setdefault("scale", "keller")
            else:
                g.setdefault("lr", 1e-3); g.setdefault("betas", (0.9, 0.95)); g.setdefault("eps", 1e-8)
                g.setdefault("weight_decay", 0.0)
        super().__init__(param_groups, dict())

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p]
                if not group["use_muon"]:
                    adamw_update_(p, state, group["lr"], group["betas"], group["eps"], group["weight_decay"])
                    continue
                g = p.grad
                if "buf" not in state:
                    state["buf"] = torch.zeros_like(g)
                buf = state["buf"]
                buf.mul_(group["momentum"]).add_(g)
                u = g.add(buf, alpha=group["momentum"]) if group["nesterov"] else buf
                u2 = u.reshape(u.shape[0], -1) if u.ndim >= 2 else u.reshape(1, -1)   # conv filters -> matrices
                O = newton_schulz5(u2, group["ns_steps"]).to(p.dtype).reshape(p.shape)
                rows, cols = u2.shape
                scale = max(1.0, rows / cols) ** 0.5 if group["scale"] == "keller" else 0.2 * math.sqrt(max(rows, cols))
                p.mul_(1 - group["lr"] * group["weight_decay"])
                p.add_(O, alpha=-group["lr"] * scale)
        return loss

def muon_for(model, lr=0.02, adam_lr=1e-3, wd=0.0, scale="keller"):
    """Split a model's params: 2-D hidden weights -> Muon, head/biases/norms -> AdamW."""
    linears = [m for m in model.modules() if isinstance(m, (nn.Linear, nn.Conv2d))]
    head = linears[-1].weight
    muon_ps = [p for p in model.parameters() if p.ndim >= 2 and p is not head]
    other = [p for p in model.parameters() if not (p.ndim >= 2 and p is not head)]
    return Muon([dict(params=muon_ps, use_muon=True, lr=lr, weight_decay=wd, scale=scale),
                 dict(params=other, use_muon=False, lr=adam_lr, weight_decay=wd)])

> **🍯 Key takeaway:** Every optimizer is $\theta \leftarrow \theta - \eta\cdot T(\text{gradient history})$: momentum *filters*, Adam *preconditions per coordinate*, Lion takes *signs*, Muon *orthogonalizes matrices*. And AdamW's decay is decoupled from the preconditioner — that's what the W means.

## Part 3 — Race #1: the 2-D arena 🏁

**Fairness rule:** every optimizer gets its *own* best learning rate from a log-spaced sweep (judged by final loss). Comparing optimizers at one shared learning rate is the #1 mistake in optimizer papers and blog posts.

For Muon we treat the point $(x,y)$ as a $1\times 2$ matrix. For a rank-1 matrix, $UV^\top$ is just the **unit-normalized** gradient direction, so Muon degenerates to *normalized-gradient descent with Nesterov momentum* — a fun sanity check on the theory.

### 🔮 Predict-then-run
1. On Rosenbrock, which optimizer gets closest to $(1,1)$ in the step budget?
2. Lion and Muon take steps of (nearly) *fixed size* $\eta$. What will their trajectories look like **near the minimum** with a constant learning rate?

In [ ]:
OPTIMIZERS_2D = {
    "SGD":      lambda p, lr: MySGD(p, lr=lr),
    "Momentum": lambda p, lr: MySGD(p, lr=lr, momentum=0.9),
    "Nesterov": lambda p, lr: MySGD(p, lr=lr, momentum=0.9, nesterov=True),
    "Adam":     lambda p, lr: MyAdam(p, lr=lr),
    "AdamW":    lambda p, lr: MyAdamW(p, lr=lr, weight_decay=0.01),
    "Lion":     lambda p, lr: Lion(p, lr=lr),
    "Muon":     lambda p, lr: Muon([dict(params=p, use_muon=True, lr=lr)]),
}
LR_GRID = np.logspace(-5, 0, 11)
STEPS_2D = 300 if FAST else 1000

def run_2d(name, make_opt, lr, steps=STEPS_2D):
    A = ARENAS[name]
    xy = nn.Parameter(torch.tensor([A["start"]], dtype=torch.float64))    # shape (1, 2)
    opt = make_opt([xy], lr)
    path, losses = [xy.detach().numpy()[0].copy()], []
    for _ in range(steps):
        loss = A["f"](xy[0, 0], xy[0, 1])
        if not torch.isfinite(loss) or loss.item() > 1e6:
            return np.array(path), np.array(losses + [np.inf])
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item()); path.append(xy.detach().numpy()[0].copy())
    return np.array(path), np.array(losses + [A["f"](*path[-1])])

results_2d = {}
t0 = time.time()
for arena in ARENAS:
    for opt_name, make in OPTIMIZERS_2D.items():
        best = None
        for lr in LR_GRID:
            path, losses = run_2d(arena, make, lr)
            score = losses[-1] if np.isfinite(losses[-1]) else np.inf
            if best is None or score < best[0]:
                best = (score, lr, path, losses)
        results_2d[(arena, opt_name)] = best
print(f"sweep done in {time.time() - t0:.1f}s")
for arena in ARENAS:
    print(f"\n{arena}:")
    for opt_name in OPTIMIZERS_2D:
        score, lr, path, _ = results_2d[(arena, opt_name)]
        print(f"  {opt_name:9s} best lr={lr:.1e}  final f={score:.2e}  final point=({path[-1][0]:.3f}, {path[-1][1]:.3f})")

In [ ]:
colors = dict(zip(OPTIMIZERS_2D, ["#1F1B16", "#3D4DB7", "#9B5DE5", "#FF6B5B", "#C2185B", "#1FA39A", "#F5B700"]))
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for col, arena in enumerate(ARENAS):
    ax = axes[0, col]; plot_landscape(ax, arena)
    for opt_name in OPTIMIZERS_2D:
        _, lr, path, _ = results_2d[(arena, opt_name)]
        ax.plot(path[:, 0], path[:, 1], "-", lw=1.6, color=colors[opt_name], label=f"{opt_name} (lr={lr:.0e})", alpha=0.9)
    ax.legend(fontsize=8, loc="lower right" if arena == "Rosenbrock" else "upper right")
    ax = axes[1, col]
    for opt_name in OPTIMIZERS_2D:
        _, _, _, losses = results_2d[(arena, opt_name)]
        ax.semilogy(np.maximum(losses, 1e-16), color=colors[opt_name], label=opt_name, lw=1.5)
    ax.set_xlabel("step"); ax.set_ylabel("f (log)"); ax.set_title(f"{arena}: loss vs step (each at its best lr)")
plt.tight_layout(); plt.show()

### 🎬 The race, animated
Same runs, replayed as a race on the Rosenbrock banana (time is log-spaced so you can see both the frantic start and the slow finish). Watch who sprints, who zig-zags, and who rattles around the finish line.

In [ ]:
from matplotlib import animation
from IPython.display import HTML, display

arena = "Rosenbrock"
fig, ax = plt.subplots(figsize=(7.5, 5.2))
plot_landscape(ax, arena)
paths = {n: results_2d[(arena, n)][2] for n in OPTIMIZERS_2D}
T = max(len(pth) for pth in paths.values())
frame_idx = np.unique(np.geomspace(1, T, 70).astype(int)) - 1
trails = {n: ax.plot([], [], "-", lw=1.6, color=colors[n], alpha=0.85)[0] for n in paths}
heads = {n: ax.plot([], [], "o", ms=8, color=colors[n], mec="k", label=n)[0] for n in paths}
ax.legend(fontsize=8, loc="lower right")
title = ax.set_title("")
def update(i):
    t = frame_idx[i]
    for n, pth in paths.items():
        q = pth[: min(t, len(pth) - 1) + 1]
        trails[n].set_data(q[:, 0], q[:, 1]); heads[n].set_data([q[-1, 0]], [q[-1, 1]])
    title.set_text(f"Optimizer race on {arena} — step {t}")
    return list(trails.values()) + list(heads.values()) + [title]
anim = animation.FuncAnimation(fig, update, frames=len(frame_idx), interval=90, blit=False)
plt.close(fig)
display(HTML(anim.to_jshtml(default_mode="once")))

**What to notice**
- **SGD** crawls along the Rosenbrock valley: its learning rate is capped by the *steepest* direction (the walls), so it's tiny along the *flattest* one (the floor). That's ill-conditioning in one picture.
- **Momentum/Nesterov** accelerate along the floor; Nesterov oscillates less.
- **Adam** divides by a running gradient scale, so it can use a *much* larger learning rate without blowing up on the steep walls. It zig-zags early but often wins these toy races outright — its per-coordinate normalization is a big deal when gradient magnitudes vary by orders of magnitude.
- **AdamW** with weight decay does *not* converge exactly to $(1,1)$ or $(3, 0.5)$: decay pulls toward the origin, so it minimizes $f(\theta) + \tfrac{\lambda}{2}\|\theta\|^2$-ish instead. On a toy function that's a bias; on a neural net it's the regularization you wanted.
- **Lion / Muon** make fixed-size steps: they make fast early progress, then **rattle around the minimum** at a radius ~ $\eta$. This is why sign/orthogonalized methods *need* learning-rate decay to converge precisely — a perfect motivation for Part 5.

> **🍯 Key takeaway:** Compare optimizers only at their *own* best learning rate — and remember that fixed-size-step methods (Lion, Muon) need LR decay to settle into a minimum.

## Part 4 — Race #2: FashionMNIST 👟👜

Toy landscapes can mislead (they're 2-D; neural nets are $10^5$–$10^{12}$-D). Now a real network: an MLP 784→256→256→10 on FashionMNIST. Each optimizer uses a sensible, commonly used learning rate (a full per-optimizer sweep is a 🏆 Boss Level). For Muon, the two hidden matrices use Muon and the head + biases use AdamW.

⚡ On CPU this takes ~1–3 minutes; GPU makes it seconds.

In [ ]:
import contextlib, io

def load_fashion_mnist():
    """torchvision -> OpenML mirror -> sklearn's bundled digits (offline last resort). Pixels in [0, 1]."""
    try:
        from torchvision import datasets
        with contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()):  # hide download bars
            tr = datasets.FashionMNIST(DATA_DIR, train=True, download=True)
            te = datasets.FashionMNIST(DATA_DIR, train=False, download=True)
        return (tr.data.float().view(-1, 784) / 255, tr.targets.clone(),
                te.data.float().view(-1, 784) / 255, te.targets.clone(), "FashionMNIST (torchvision)")
    except Exception as e:
        print(f"⚠️ torchvision download failed ({type(e).__name__}); trying the OpenML mirror...")
    try:
        from sklearn.datasets import fetch_openml
        Xo, yo = fetch_openml("Fashion-MNIST", version=1, return_X_y=True, as_frame=False, data_home=DATA_DIR)
        Xo, yo = torch.tensor(Xo, dtype=torch.float32) / 255, torch.tensor(yo.astype(np.int64))
        return Xo[:60000], yo[:60000], Xo[60000:], yo[60000:], "FashionMNIST (OpenML)"
    except Exception as e:
        print(f"⚠️ OpenML failed too ({type(e).__name__}); using sklearn's bundled 8x8 digits upsampled to 28x28.")
    from sklearn.datasets import load_digits
    d = load_digits()
    X = F.interpolate(torch.tensor(d.images, dtype=torch.float32).unsqueeze(1) / 16, size=(28, 28),
                      mode="bilinear", align_corners=False).view(-1, 784)
    y = torch.tensor(d.target)
    perm = torch.randperm(len(X), generator=torch.Generator().manual_seed(0))
    X, y = X[perm], y[perm]
    return X[:1500], y[:1500], X[1500:], y[1500:], "digits (offline fallback)"

Xtr, ytr, Xte, yte, DATASET = load_fashion_mnist()
MEAN, STD = Xtr.mean().item(), Xtr.std().item()        # ≈ 0.286 / 0.353 for FashionMNIST
Xtr, Xte = (Xtr - MEAN) / STD, (Xte - MEAN) / STD
print("dataset:", DATASET)
if FAST:
    Xtr, ytr, Xte, yte = Xtr[:6000], ytr[:6000], Xte[:2000], yte[:2000]
Xtr, ytr, Xte, yte = Xtr.to(DEVICE), ytr.to(DEVICE), Xte.to(DEVICE), yte.to(DEVICE)
print("train", tuple(Xtr.shape), "test", tuple(Xte.shape))

def make_mlp(width=256):
    return nn.Sequential(nn.Linear(784, width), nn.ReLU(), nn.Linear(width, width), nn.ReLU(), nn.Linear(width, 10))

@torch.no_grad()
def accuracy(model, X, y, bs=4096):
    model.eval()
    correct = sum((model(X[i:i + bs]).argmax(1) == y[i:i + bs]).sum().item() for i in range(0, len(X), bs))
    model.train()
    return correct / len(X)

def train(make_opt, steps, model_fn=make_mlp, sched=None, bs=128, seed=0):
    """Generic loop. `sched(step) -> multiplier` scales every group's base lr."""
    seed_everything(seed)
    model = model_fn().to(DEVICE)
    opt = make_opt(model)
    for g in opt.param_groups:
        g["base_lr"] = g["lr"]
    gen = torch.Generator().manual_seed(seed)
    losses, lrs = [], []
    for step in range(steps):
        if sched is not None:
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * sched(step)
        idx = torch.randint(0, len(Xtr), (bs,), generator=gen).to(DEVICE)
        loss = F.cross_entropy(model(Xtr[idx]), ytr[idx])
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        losses.append(loss.item()); lrs.append(opt.param_groups[0]["lr"])
    return dict(losses=np.array(losses), lrs=np.array(lrs), test_acc=accuracy(model, Xte, yte), model=model)

In [ ]:
STEPS_FM = 150 if FAST else 1200   # 1200 steps x 128 = ~2.5 epochs
FM_OPTS = {
    "SGD (lr=0.1)":              lambda m: MySGD(m.parameters(), lr=0.1),
    "Momentum (0.05, 0.9)":      lambda m: MySGD(m.parameters(), lr=0.05, momentum=0.9),
    "Nesterov (0.05, 0.9)":      lambda m: MySGD(m.parameters(), lr=0.05, momentum=0.9, nesterov=True),
    "Adam (1e-3)":               lambda m: MyAdam(m.parameters(), lr=1e-3),
    "AdamW (1e-3, wd=0.05)":     lambda m: MyAdamW(m.parameters(), lr=1e-3, weight_decay=0.05),
    "Lion (1e-4, wd=0.5)":       lambda m: Lion(m.parameters(), lr=1e-4, weight_decay=0.5),
    "Muon (0.02) + AdamW head":  lambda m: muon_for(m, lr=0.02, adam_lr=1e-3),
}
fm_results = {}
for name, make in FM_OPTS.items():
    t0 = time.time()
    fm_results[name] = train(make, STEPS_FM)
    print(f"{name:26s} test acc {fm_results[name]['test_acc']:.2%}   final train loss (EMA) "
          f"{ema(fm_results[name]['losses'])[-1]:.3f}   [{time.time() - t0:.1f}s]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw=dict(width_ratios=[2, 1]))
cols7 = ["#1F1B16", "#3D4DB7", "#9B5DE5", "#FF6B5B", "#C2185B", "#1FA39A", "#F5B700"]
for c, (name, r) in zip(cols7, fm_results.items()):
    axes[0].plot(ema(r["losses"]), label=name, color=c, lw=2 if "Muon" in name else 1.3)
axes[0].set_yscale("log"); axes[0].set_xlabel("step"); axes[0].set_ylabel("train loss (EMA, log)")
axes[0].set_title("FashionMNIST MLP — training loss"); axes[0].legend(fontsize=8)
names = list(fm_results)
axes[1].barh(range(len(names)), [fm_results[n]["test_acc"] * 100 for n in names], color=cols7)
axes[1].set_yticks(range(len(names))); axes[1].set_yticklabels([n.split(" (")[0] for n in names])
lo = min(fm_results[n]["test_acc"] for n in names) * 100
hi = max(fm_results[n]["test_acc"] for n in names) * 100
axes[1].set_xlim(lo - 3, min(100, hi + 3)); axes[1].set_xlabel("test accuracy (%)"); axes[1].set_title("Test accuracy")
plt.tight_layout(); plt.show()

Typical outcome (exact numbers vary by seed/hardware): plain SGD is slowest; momentum methods, Adam(W) and Lion cluster together; **Muon** is usually at or near the front in *training loss per step*. On a small MLP the differences are modest — Muon's big wins show up on transformers where hidden matrices dominate the parameter count and step budgets are tight. Remember: **one seed and one learning rate per optimizer is an anecdote, not a benchmark.**

> **🍯 Key takeaway:** On a real network the gaps between well-tuned optimizers are smaller than on toy problems; Muon's orthogonalized updates gave the fastest training-loss drop here, and its advantage is largest when hidden matrices dominate the model.

## Part 5 — Learning-rate schedules: warmup + cosine vs WSD 📈

**Why warm up?** At step 0 Adam's second-moment estimate $\hat v$ is based on very few samples and the network is far from any reasonable region; large early steps can blow up activations (especially in post-norm transformers). A short linear warmup (typically 1–5% of training) is cheap insurance (Goyal et al., 2017, popularized it for large-batch training).

**Warmup + cosine** (Loshchilov & Hutter's SGDR, without restarts): after warmup,
$$\eta_t = \eta_{\min} + \tfrac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\frac{\pi (t - t_w)}{T - t_w}\right).$$
The catch: **you must know $T$ in advance**. Want to train longer? The schedule is wrong for every step already taken.

**WSD — warmup, stable, decay** (used e.g. in MiniCPM, Hu et al. 2024; analysed by Hägele et al. 2024): warm up, hold $\eta_{\max}$ *constant* for most of training, then decay quickly (e.g. the last 10–20%). Loss during the stable phase looks worse — then drops sharply in the cooldown, typically matching cosine. Payoff: you can **branch** a cooldown from any stable-phase checkpoint, so one long run yields models for many compute budgets (great for scaling-law studies) and training can be extended indefinitely.

In [ ]:
def warmup_cosine(total, warmup_frac=0.05, min_ratio=0.0):
    w = max(1, int(warmup_frac * total))
    def f(step):
        if step < w:
            return (step + 1) / w
        prog = (step - w) / max(1, total - w)
        return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * prog))
    return f

### 🧪 Your Turn #4 — implement WSD
Linear warmup for `warmup_frac` of training, constant 1.0, then a **linear** decay to `min_ratio` over the final `decay_frac` of steps.

**Hint:** three `if` branches. The decay progress is `(step - decay_start) / decay_steps`.

In [ ]:
# 🧪 YOUR TURN
def wsd(total, warmup_frac=0.05, decay_frac=0.2, min_ratio=0.0):
    w = max(1, int(warmup_frac * total))
    d = max(1, int(decay_frac * total))
    def f(step):
        # TODO: warmup / stable / decay
        return 1.0
    return f

In [ ]:
# ✅ Solution (peek only after trying!)
def wsd(total, warmup_frac=0.05, decay_frac=0.2, min_ratio=0.0):
    w = max(1, int(warmup_frac * total))
    d = max(1, int(decay_frac * total))
    def f(step):
        if step < w:
            return (step + 1) / w
        if step < total - d:
            return 1.0
        prog = (step - (total - d)) / d
        return 1.0 - (1.0 - min_ratio) * prog
    return f

In [ ]:
# ✔️ Checkpoint — the unchanged stub (always 1.0) fails this
s = wsd(1000)
assert abs(s(0) - 1 / 50) < 1e-9 and s(49) == 1.0 and s(500) == 1.0 and s(799) == 1.0
assert abs(s(900) - 0.5) < 1e-9 and s(999) < 0.01
print("✔️ WSD schedule looks right")

### 🔮 Predict-then-run
We train the same AdamW MLP with (a) constant LR, (b) warmup+cosine, (c) WSD. Sketch the three loss curves before running: which is *lowest at 60% of training*? Which is lowest at the *end*?

In [ ]:
STEPS_S = 200 if FAST else 1500
LR_S = 2e-3
scheds = {"constant": lambda s: 1.0, "warmup+cosine": warmup_cosine(STEPS_S), "WSD": wsd(STEPS_S)}
sched_results = {}
for name, sch in scheds.items():
    sched_results[name] = train(lambda m: MyAdamW(m.parameters(), lr=LR_S, weight_decay=0.05), STEPS_S, sched=sch)
    print(f"{name:14s} test acc {sched_results[name]['test_acc']:.2%}  final EMA loss {ema(sched_results[name]['losses'])[-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for c, (name, r) in zip([HIVE[1], HIVE[4], HIVE[0]], sched_results.items()):
    axes[0].plot(r["lrs"], label=name, color=c, lw=2)
    axes[1].plot(ema(r["losses"], 0.98), label=f"{name} (test {r['test_acc']:.1%})", color=c, lw=1.8)
axes[0].set_title("learning-rate schedules"); axes[0].set_xlabel("step"); axes[0].legend()
axes[1].set_title("train loss (EMA)"); axes[1].set_xlabel("step"); axes[1].set_yscale("log"); axes[1].legend()
axes[1].axvspan(int(0.8 * STEPS_S), STEPS_S, color=HIVE[0], alpha=0.12)
axes[1].text(int(0.8 * STEPS_S), axes[1].get_ylim()[1] * 0.8, " WSD cooldown", fontsize=9)
plt.tight_layout(); plt.show()

You should see WSD tracking the constant-LR curve (above cosine) during the stable phase, then **dropping sharply** during the cooldown. A large part of "the benefit of LR decay" is simply *removing the noise floor* that a large constant step size imposes — the same rattling you saw with Lion/Muon in Part 3.

> **🍯 Key takeaway:** LR decay mostly removes the *noise floor* of a large step size. WSD gets the same benefit as cosine while letting you choose when to stop *after* the fact.

## Part 6 — The LR range test 🔎

How do you pick $\eta_{\max}$ without a full sweep? Smith's **LR range test** (2017): start tiny, multiply the LR by a constant factor every step (exponential ramp), record the (smoothed) loss, and stop when it explodes. The loss-vs-LR curve has three regions: *too small* (flat), *just right* (steep descent), *too big* (blow-up).

A common heuristic: choose an LR about **10× below the LR at the loss minimum** (or near the steepest descent point). It's a heuristic — it measures the *start* of training, and the best final LR for long runs is often smaller — but it is cheap (one short run) and catches order-of-magnitude mistakes.

In [ ]:
def lr_range_test(make_opt, lr_min=1e-6, lr_max=1.0, steps=200, bs=128, seed=0, diverge=4.0):
    seed_everything(seed)
    model = make_mlp().to(DEVICE)
    opt = make_opt(model.parameters(), lr_min)
    gamma = (lr_max / lr_min) ** (1 / steps)
    gen = torch.Generator().manual_seed(seed)
    lrs, losses, best, avg = [], [], float("inf"), 0.0
    for k in range(steps):
        lr = lr_min * gamma ** k
        for g in opt.param_groups:
            g["lr"] = lr
        idx = torch.randint(0, len(Xtr), (bs,), generator=gen).to(DEVICE)
        loss = F.cross_entropy(model(Xtr[idx]), ytr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        avg = 0.9 * avg + 0.1 * loss.item()
        smooth = avg / (1 - 0.9 ** (k + 1))           # bias-corrected EMA (like Adam's m-hat)
        lrs.append(lr); losses.append(smooth)
        best = min(best, smooth)
        if smooth > diverge * best or not math.isfinite(smooth):
            break
    return np.array(lrs), np.array(losses)

RT_STEPS = 120 if FAST else 300
rt = {"AdamW": lr_range_test(lambda p, lr: MyAdamW(p, lr=lr), steps=RT_STEPS),
      "SGD+momentum": lr_range_test(lambda p, lr: MySGD(p, lr=lr, momentum=0.9), steps=RT_STEPS, lr_max=10.0)}

fig, ax = plt.subplots(figsize=(9, 4.5))
for c, (name, (lrs, losses)) in zip([HIVE[2], HIVE[4]], rt.items()):
    ax.semilogx(lrs, losses, label=name, color=c, lw=2)
    lr_at_min = lrs[np.argmin(losses)]
    ax.axvline(lr_at_min / 10, color=c, ls="--", lw=1)
    print(f"{name:13s}: loss minimum at lr≈{lr_at_min:.1e}  ->  suggested lr ≈ {lr_at_min / 10:.1e}")
ax.set_xlabel("learning rate (log)"); ax.set_ylabel("smoothed loss"); ax.set_ylim(0, 3.5)
ax.set_title("LR range test (dashed = suggested LR, 10x below the minimum)"); ax.legend()
plt.show()

> **🍯 Key takeaway:** One short run with an exponentially growing LR tells you the right order of magnitude — the most important hyperparameter, found in under a minute.

## Part 7 — Why residuals + normalization + init make depth trainable 🏗️

Optimizers can only do so much if the network itself turns gradients into dust (vanishing) or fireworks (exploding). Three architectural ingredients fix that:

**1. Initialization.** For $h_{l+1} = \phi(W h_l)$ with $W_{ij}\sim\mathcal N(0,\sigma^2)$, $\mathrm{Var}[W h] = n_{\text{in}}\sigma^2\,\mathbb E[h^2]$. To keep the signal scale constant you need $\sigma^2 = 1/n_{\text{in}}$ (Glorot/LeCun-style for linear/tanh) or $2/n_{\text{in}}$ for ReLU, which zeros half its inputs (He et al. 2015). Off by a factor $k$ per layer ⇒ off by $k^L$ after $L$ layers.

**2. Residual connections** $h_{l+1} = h_l + F(h_l)$ (He et al. 2016). The Jacobian is $I + \partial F/\partial h$: there's always an identity path, so gradients reach early layers undiminished. But the *forward* variance adds up: $\mathrm{Var}[h_L] \approx \mathrm{Var}[h_0] + \sum_l \mathrm{Var}[F_l]$, which can explode if each branch is proportional to its input. (GPT-2 scales residual-branch output weights by $1/\sqrt{N_{\text{layers}}}$ for this reason.)

**3. Normalization.** **LayerNorm** (Ba et al. 2016): $\ \mathrm{LN}(x) = \gamma\odot\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}} + \beta$ over the feature dimension. **RMSNorm** (Zhang & Sennrich 2019) drops the mean-centering and $\beta$: $\ \mathrm{RMSNorm}(x) = \gamma \odot \frac{x}{\sqrt{\frac1d\sum_i x_i^2 + \epsilon}}$ — cheaper, and empirically as good, so it's the default in Llama-family LLMs. In a **pre-norm** block, $h_{l+1} = h_l + F(\mathrm{Norm}(h_l))$, each branch sees a unit-scale input whatever the depth, so its output scale is bounded; the residual stream grows only ~linearly (~$\sqrt L$ in std) instead of exponentially. Pre-norm transformers are much less sensitive to warmup than the original post-norm design (Xiong et al. 2020).

### 🧪 Your Turn #5 — RMSNorm from scratch
**Hint:** `x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight`.

In [ ]:
# 🧪 YOUR TURN
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return x   # TODO

In [ ]:
# ✅ Solution (peek only after trying!)
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

In [ ]:
# ✔️ Checkpoint — unit RMS with γ=1, and a match with PyTorch's built-in (torch >= 2.4)
x = torch.randn(4, 7, 64) * 3 + 1
assert torch.allclose(RMSNorm(64)(x).pow(2).mean(-1), torch.ones(4, 7), atol=1e-3), "output RMS should be 1 when γ=1"
ours = RMSNorm(64); nn.init.normal_(ours.weight)
if hasattr(nn, "RMSNorm"):
    ref = nn.RMSNorm(64, eps=1e-6); ref.weight.data.copy_(ours.weight.data)
    assert torch.allclose(ours(x), ref(x), atol=1e-5)
    print("✔️ RMSNorm matches torch.nn.RMSNorm")
ln = nn.LayerNorm(64, elementwise_affine=False)
print("LayerNorm output: mean≈%.1e, std≈%.3f | RMSNorm(γ=1) output RMS≈%.3f" % (
    ln(x).mean().item(), ln(x).std(-1, unbiased=False).mean().item(), RMSNorm(64)(x).pow(2).mean(-1).sqrt().mean().item()))

In [ ]:
class Block(nn.Module):
    def __init__(self, d, residual, norm):
        super().__init__()
        self.norm = RMSNorm(d) if norm else nn.Identity()
        self.fc = nn.Linear(d, d)
        self.residual = residual
    def forward(self, x):
        h = torch.relu(self.fc(self.norm(x)))
        return x + h if self.residual else h

class DeepNet(nn.Module):
    def __init__(self, depth=48, d=128, residual=False, norm=False, he_init=False):
        super().__init__()
        self.inp = nn.Linear(784, d)
        self.blocks = nn.ModuleList([Block(d, residual, norm) for _ in range(depth)])
        self.out_norm = RMSNorm(d) if norm else nn.Identity()
        self.head = nn.Linear(d, 10)
        if he_init:
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(m.weight, nonlinearity="relu"); nn.init.zeros_(m.bias)
    def forward(self, x, record=None):
        h = self.inp(x)
        for b in self.blocks:
            h = b(h)
            if record is not None:
                record.append(h.detach().pow(2).mean().sqrt().item())
        return self.head(self.out_norm(h))

DEPTH = 24 if FAST else 48
VARIANTS = {
    "plain, default init":          dict(),
    "plain, He init":               dict(he_init=True),
    "residual, no norm":            dict(residual=True),
    "residual + pre-RMSNorm":       dict(residual=True, norm=True),
}
seed_everything(0)
xb, yb = Xtr[:512], ytr[:512]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for c, (name, kw) in zip([HIVE[1], HIVE[4], HIVE[2], HIVE[0]], VARIANTS.items()):
    torch.manual_seed(0)
    net = DeepNet(DEPTH, **kw).to(DEVICE)
    rms = []
    loss = F.cross_entropy(net(xb, record=rms), yb)
    loss.backward()
    gnorms = [b.fc.weight.grad.norm().item() for b in net.blocks]
    axes[0].semilogy(range(1, DEPTH + 1), rms, label=name, color=c, lw=2)
    axes[1].semilogy(range(1, DEPTH + 1), np.maximum(gnorms, 1e-30), label=name, color=c, lw=2)
axes[0].set_title(f"forward: activation RMS per block (depth {DEPTH}, at init)"); axes[0].set_xlabel("block")
axes[1].set_title("backward: ‖∂L/∂W‖ per block (at init)"); axes[1].set_xlabel("block")
axes[0].legend(fontsize=9); plt.tight_layout(); plt.show()

### 🔮 Predict-then-run
Now we train all four deep nets with the *same* optimizer (AdamW, lr 1e-3) for a few hundred steps. Which ones will learn at all? Adam normalizes gradient *scale* per coordinate — does that rescue the vanishing-gradient network?

In [ ]:
STEPS_D = 60 if FAST else 400
fig, ax = plt.subplots(figsize=(9, 4.5))
for c, (name, kw) in zip([HIVE[1], HIVE[4], HIVE[2], HIVE[0]], VARIANTS.items()):
    r = train(lambda m: MyAdamW(m.parameters(), lr=1e-3, weight_decay=0.0), STEPS_D,
              model_fn=lambda kw=kw: DeepNet(DEPTH, **kw))
    ax.plot(ema(r["losses"], 0.9), label=f"{name} (test {r['test_acc']:.1%})", color=c, lw=2)
    print(f"{name:24s} test acc {r['test_acc']:.2%}")
ax.set_yscale("log"); ax.set_xlabel("step"); ax.set_ylabel("train loss (EMA)")
ax.set_title(f"Training {DEPTH}-block MLPs with AdamW"); ax.legend(fontsize=9); plt.show()

**Takeaways.** With PyTorch's default `nn.Linear` init (variance $\tfrac{1}{3n_{\text{in}}}$) and ReLU, a plain deep stack shrinks the signal at every layer, and the gradient for early layers is astronomically small. He init fixes the *forward* scale for this simple plain net, but it is fragile (any change of activation, width or depth breaks the balance). The residual net without normalization blows up activations exponentially in depth. **Residual + pre-norm** keeps both forward activations and backward gradients in a narrow, healthy band — which is exactly why every modern transformer block looks like `x = x + Attn(Norm(x)); x = x + MLP(Norm(x))`. Adam can rescale tiny gradients, but it can't recover information that a vanishing forward pass has already destroyed.

> **🍯 Key takeaway:** Trainability at depth is an *architecture* property: residual paths keep gradients flowing, (pre-)normalization keeps every block's input at unit scale, and init sets the starting scale. No optimizer can rescue a dead signal path.

## 🏆 Boss Level
1. **Fair benchmark.** For each optimizer in Part 4, sweep the LR over a log grid (and 3 seeds) and report mean ± std of the best test accuracy. Does the ranking survive? (See Schmidt et al. 2021 and Dahl et al. 2023 for why this matters.)
2. **Muon on a CNN / tiny transformer.** Swap the MLP for a small CNN (Muon treats each conv kernel as a `(out, in·k·k)` matrix — already supported by our `reshape`) or a 2-layer transformer. Compare `scale="keller"` vs `scale="moonlight"` (with AdamW's LR and weight decay reused for the Muon group).
3. **Schedule-free.** Implement Schedule-Free AdamW (Defazio et al. 2024), which removes the need to know the training length, and race it against WSD.

## 🍯 Key Takeaways
- Every optimizer is `θ ← θ − η · T(gradient history)`: momentum **filters**, Adam **preconditions per coordinate**, Lion takes **signs**, Muon **orthogonalizes matrices** (steepest descent in the spectral norm).
- **AdamW ≠ Adam + L2**: decoupled decay shrinks weights directly instead of routing through $1/\sqrt{\hat v}$ — and your implementation matched `torch.optim.AdamW`.
- **Newton–Schulz** approximates $UV^\top$ with a handful of matmuls; use Muon on 2-D hidden weights and AdamW for embeddings, heads, biases and norms.
- Fixed-size-step methods rattle at a radius ~η — **LR decay** removes that noise floor; **WSD** lets you decide the training length late.
- Benchmark optimizers fairly: each at its own tuned LR, over several seeds.
- The **LR range test** finds the right LR order of magnitude in one short run.
- **Residual + pre-norm + sane init** make depth trainable; the optimizer can't fix a broken signal path.

## 🎤 Explain it to a friend
You're hiking down a foggy mountain valley that is very steep side-to-side but gently sloping along its length. Explain to a friend why taking steps proportional to the slope (SGD) makes you bounce between the walls while barely moving forward, how keeping some running momentum helps, and why "always take a step of the same length in the best direction" (Lion/Muon) gets you down fast but leaves you pacing around the bottom unless you shorten your stride at the end.

## 💼 Interview corner
**1. What is the difference between Adam with L2 regularization and AdamW?**
<details><summary>Answer</summary>

Adam+L2 adds $\lambda\theta$ to the gradient, so the decay term is divided by $\sqrt{\hat v}$ — parameters with large historical gradients get *less* regularization. AdamW applies $\theta \leftarrow \theta - \eta\lambda\theta$ separately from the adaptive update, so every weight decays at the same relative rate. With plain SGD the two coincide; with adaptive methods they don't, and AdamW typically generalizes better and makes $\lambda$ easier to tune.
</details>

**2. What does Muon do, and which parameters should it be applied to?**
<details><summary>Answer</summary>

Muon keeps SGD-style (Nesterov) momentum for each 2-D weight matrix and replaces the update by an approximate orthogonalization $UV^\top$ of the momentum (computed with a few Newton–Schulz iterations), i.e. it sets all singular values of the update to ~1 — steepest descent under the spectral norm. It's meant for hidden-layer matrices (conv kernels reshaped to matrices); embeddings, the output head, biases and norm gains are usually trained with AdamW.
</details>

**3. Why warm up the learning rate, and why might you pick WSD over cosine?**
<details><summary>Answer</summary>

Early in training, Adam's second-moment estimates are noisy and the network is in a poorly-conditioned region, so large steps can blow up activations; a short linear warmup avoids this. Cosine needs the total step count fixed up front. WSD holds the LR constant and decays only at the end, so you can branch a cooldown from any checkpoint (one run → models for many budgets) or keep training longer — with final loss typically comparable to cosine.
</details>

## 🐝 Fun fact
Muon first made its name not in a journal but on a public leaderboard: it powered new records in the community "NanoGPT speedrun" (training a GPT-2-small-sized model to a target validation loss as fast as possible) in late 2024 — before being scaled up to multi-billion-parameter LLMs in 2025.

## 📚 Further reading
- D. P. Kingma, J. Ba, "Adam: A Method for Stochastic Optimization", ICLR 2015.
- I. Loshchilov, F. Hutter, "Decoupled Weight Decay Regularization", ICLR 2019. — and "SGDR: Stochastic Gradient Descent with Warm Restarts", ICLR 2017.
- I. Sutskever, J. Martens, G. Dahl, G. Hinton, "On the importance of initialization and momentum in deep learning", ICML 2013.
- X. Chen et al., "Symbolic Discovery of Optimization Algorithms" (Lion), NeurIPS 2023.
- K. Jordan et al., "Muon: An optimizer for hidden layers in neural networks", blog post, 2024 (kellerjordan.github.io/posts/muon).
- J. Liu et al. (Moonshot AI), "Muon is Scalable for LLM Training", arXiv:2502.16982, 2025.
- J. Bernstein, L. Newhouse, "Old Optimizer, New Norm: An Anthology", arXiv:2409.20325, 2024.
- L. N. Smith, "Cyclical Learning Rates for Training Neural Networks", WACV 2017.
- P. Goyal et al., "Accurate, Large Minibatch SGD: Training ImageNet in 1 Hour", arXiv:1706.02677, 2017.
- S. Hu et al., "MiniCPM: Unveiling the Potential of Small Language Models with Scalable Training Strategies", 2024.
- A. Hägele et al., "Scaling Laws and Compute-Optimal Training Beyond Fixed Training Durations", NeurIPS 2024.
- A. Defazio et al., "The Road Less Scheduled", NeurIPS 2024.
- R. M. Schmidt, F. Schneider, P. Hennig, "Descending through a Crowded Valley — Benchmarking Deep Learning Optimizers", ICML 2021.
- G. E. Dahl et al., "Benchmarking Neural Network Training Algorithms", arXiv:2306.07179, 2023.
- K. He, X. Zhang, S. Ren, J. Sun, "Delving Deep into Rectifiers" (ICCV 2015) and "Deep Residual Learning for Image Recognition" (CVPR 2016).
- J. L. Ba, J. R. Kiros, G. E. Hinton, "Layer Normalization", arXiv 2016; B. Zhang, R. Sennrich, "Root Mean Square Layer Normalization", NeurIPS 2019.
- R. Xiong et al., "On Layer Normalization in the Transformer Architecture", ICML 2020.